# Loan Approval Prediction — Exploratory Data Analysis

This notebook performs the full EDA described in the project spec: dataset overview, missing-value analysis, distribution plots, and categorical/numeric relationships with the target, `Loan_Status`.

> **Note on the data:** the original Kaggle *Loan Prediction Dataset* (by `altruistdelhite04`) could not be downloaded in this offline environment, so `data/train.csv` is a synthetically generated dataset with the same columns, size, missing-value pattern, and the well-documented `Credit_History → Loan_Status` relationship. Swap in the real Kaggle CSV and every cell below still runs unchanged.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import load_data, dataset_overview, missing_value_report, handle_missing_values, remove_duplicates, detect_outliers_iqr
from eda import run_full_eda, eda_summary_text

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## Step 1 — Load Dataset & Overview

In [ ]:
df = load_data('../data/train.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all')

## Step 2 — Missing Values

`Credit_History`, `Self_Employed`, and `LoanAmount` carry the most missing values. Categorical gaps are filled with the column **mode**; numeric gaps with the column **median** (robust to the right-skewed income/loan-amount distributions).

In [ ]:
missing_value_report(df)

In [ ]:
df = remove_duplicates(df)
df = handle_missing_values(df)
assert df.isnull().sum().sum() == 0
print('Missing values after cleaning: 0')

## Step 3 — Outlier Check (IQR method)

We report outliers for transparency but do **not** drop them — unusually high income or loan amounts are legitimate signal in a credit-risk context, not data-entry errors.

In [ ]:
detect_outliers_iqr(df)

## Step 4 — Loan Approval Distribution

The target is imbalanced: a majority of applications are approved. This motivates `class_weight='balanced'` in the Logistic Regression model trained later in `main.py`, and stratified train/test splitting.

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(data=df, x='Loan_Status', hue='Loan_Status', palette='Set2', legend=False, ax=ax)
ax.set_title('Loan Approval Distribution')
plt.show()
df['Loan_Status'].value_counts(normalize=True)

## Step 5 — Income & Loan Amount Distributions

Both `ApplicantIncome` and `LoanAmount` are right-skewed with a long high-value tail — typical of income data — which is why median (not mean) imputation was used above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df['ApplicantIncome'], kde=True, ax=axes[0], color='#2563eb')
axes[0].set_title('Applicant Income Distribution')
sns.histplot(df['LoanAmount'].dropna(), kde=True, ax=axes[1], color='#16a34a')
axes[1].set_title('Loan Amount Distribution')
plt.tight_layout(); plt.show()

## Step 6 — Boxplots (Outlier Visualization)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14,4))
sns.boxplot(y=df['ApplicantIncome'], ax=axes[0], color='#93c5fd')
sns.boxplot(y=df['CoapplicantIncome'], ax=axes[1], color='#fca5a5')
sns.boxplot(y=df['LoanAmount'], ax=axes[2], color='#86efac')
for ax, title in zip(axes, ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']):
    ax.set_title(title)
plt.tight_layout(); plt.show()

## Step 7 — Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap (Numeric Features)')
plt.tight_layout(); plt.show()

## Step 8 — Categorical Features vs Loan Status

Grid of count plots: Education, Gender, Property Area, Credit History, Dependents, Married, Self Employed — each split by approval outcome. **Credit History** shows by far the sharpest separation between the two bars; the rest show comparatively modest differences.

In [ ]:
cat_pairs = [
    ('Education', 'Education vs Loan Status'),
    ('Gender', 'Gender vs Loan Status'),
    ('Property_Area', 'Property Area vs Loan Status'),
    ('Credit_History', 'Credit History vs Loan Status'),
    ('Dependents', 'Dependents vs Loan Status'),
    ('Married', 'Married vs Loan Status'),
    ('Self_Employed', 'Self Employed vs Loan Status'),
]
fig, axes = plt.subplots(3, 3, figsize=(15,12))
axes = axes.flatten()
for i, (col, title) in enumerate(cat_pairs):
    sns.countplot(data=df, x=col, hue='Loan_Status', palette='Set2', ax=axes[i])
    axes[i].set_title(title)
    axes[i].tick_params(axis='x', rotation=20)
for j in range(len(cat_pairs), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.show()

## Step 9 — Income vs Loan Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.boxplot(data=df, x='Loan_Status', y='ApplicantIncome', hue='Loan_Status', palette='Set2', legend=False, ax=axes[0])
axes[0].set_title('Applicant Income vs Loan Status')
sns.boxplot(data=df, x='Loan_Status', y='CoapplicantIncome', hue='Loan_Status', palette='Set2', legend=False, ax=axes[1])
axes[1].set_title('Coapplicant Income vs Loan Status')
plt.tight_layout(); plt.show()

## Step 10 — EDA Summary

In [ ]:
print(eda_summary_text(df))

## Key Takeaways

1. **Credit_History is the dominant predictor** — applicants with a clean credit history are approved at a dramatically higher rate than those without one. This single feature carries far more separating power than any other column, categorical or numeric.
2. **Income and loan amount are right-skewed**, motivating median imputation and engineered ratio features (`Total_Income`, `Income_to_Loan_Ratio`, `EMI_to_Income`) that better capture affordability than the raw columns alone.
3. **Demographic fields (Gender, Married, Self_Employed, Education) show comparatively weak association** with approval outcome in this data — a pattern the Logistic Regression coefficients in `main.py` confirm numerically.
4. **The target is moderately imbalanced** (~70/30 approved/rejected), which is why the model in this project uses `class_weight='balanced'` and a stratified train/test split.

Continue to `../main.py` for the full modeling pipeline: feature engineering → encoding → scaling → Logistic Regression → evaluation → feature-importance / odds-ratio analysis → sample prediction.